# Whisper Learning Colab GPU Worker

This notebook runs only the remote Whisper transcription worker for the local Whisper Learning app. The local app still records audio, trims attempts, scores transcripts, saves `runs/<attemptId>/result.json`, keeps history, and exports reports.

Important limitations:
- Colab Free GPU availability is not guaranteed.
- The runtime may disconnect or sleep.
- The public URL changes each session.
- Audio is sent to this remote Colab runtime. Do not use this mode for private or sensitive recordings.

In [ ]:
!nvidia-smi || true
!apt-get -y update >/dev/null
!apt-get -y install ffmpeg >/dev/null
!pip -q install openai-whisper flask flask-cors requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!which cloudflared
!cloudflared --version

In [ ]:
import os
import tempfile
import threading
import time

import torch
import whisper
from flask import Flask, jsonify, request
from flask_cors import CORS

DEFAULT_MODEL = os.environ.get("WHISPER_MODEL", "large-v3-turbo")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_CACHE = {}

def get_model(model_name):
    requested = model_name or DEFAULT_MODEL
    cache_key = (requested, DEVICE)
    if cache_key not in MODEL_CACHE:
        try:
            MODEL_CACHE[cache_key] = whisper.load_model(requested, device=DEVICE)
        except Exception:
            if requested == "large":
                raise
            fallback_key = ("large", DEVICE)
            if fallback_key not in MODEL_CACHE:
                MODEL_CACHE[fallback_key] = whisper.load_model("large", device=DEVICE)
            return MODEL_CACHE[fallback_key], "large"
    return MODEL_CACHE[cache_key], requested

print(f"Loading Whisper model {DEFAULT_MODEL} on {DEVICE}...")
model, loaded_name = get_model(DEFAULT_MODEL)
print(f"Loaded {loaded_name} on {DEVICE}.")

In [ ]:
app = Flask(__name__)
CORS(app)

@app.get("/health")
def health():
    return jsonify({"ok": True, "model": loaded_name, "device": DEVICE})

@app.post("/transcribe")
def transcribe():
    started = time.perf_counter()
    upload = request.files.get("audio")
    if upload is None:
        return jsonify({"status": "failed", "transcript": "", "error": "Missing multipart field: audio", "model": loaded_name, "device": DEVICE, "timeSec": 0}), 400

    language = (request.form.get("language") or "").strip() or None
    requested_model = (request.form.get("model") or DEFAULT_MODEL).strip()
    fast_mode = (request.form.get("fastMode") or "true").lower() == "true"

    suffix = os.path.splitext(upload.filename or "audio.wav")[1] or ".wav"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        upload.save(tmp.name)
        audio_path = tmp.name

    try:
        active_model, active_name = get_model(requested_model)
        options = {
            "language": language,
            "fp16": DEVICE == "cuda",
            "temperature": 0,
            "condition_on_previous_text": False,
            "word_timestamps": False,
        }
        if fast_mode:
            options.update({"beam_size": 1, "best_of": 1})
        else:
            options.update({"beam_size": 5, "best_of": 5})
        result = active_model.transcribe(audio_path, **options)
        transcript = (result.get("text") or "").strip()
        return jsonify({
            "status": "ok",
            "transcript": transcript,
            "model": active_name,
            "device": DEVICE,
            "timeSec": round(time.perf_counter() - started, 3),
        })
    except Exception as exc:
        return jsonify({
            "status": "failed",
            "transcript": "",
            "error": str(exc),
            "model": requested_model,
            "device": DEVICE,
            "timeSec": round(time.perf_counter() - started, 3),
        }), 500
    finally:
        try:
            os.remove(audio_path)
        except OSError:
            pass

In [ ]:
import requests
import re
import subprocess

PORT = 7860
if "server_thread" not in globals() or not server_thread.is_alive():
    server_thread = threading.Thread(target=lambda: app.run(host="0.0.0.0", port=PORT), daemon=True)
    server_thread.start()
    time.sleep(2)
else:
    print("Flask server is already running.")
print("Testing local worker health:")
local_health = None
for attempt in range(10):
    try:
        local_health = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=10).json()
        break
    except Exception as exc:
        print(f"Local health check attempt {attempt + 1}/10 failed: {exc}")
        time.sleep(1)
if local_health is None:
    raise RuntimeError("Local Flask health check failed. The worker server did not start correctly.")
print(local_health)
if "cloudflared" in globals() and cloudflared.poll() is None:
    cloudflared.terminate()
    time.sleep(1)
cloudflared = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
        "--loglevel",
        "info",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
public_url = None
print("Starting Cloudflare tunnel...")
deadline = time.time() + 120
while time.time() < deadline:
    line = cloudflared.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    print(line.rstrip())
    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    print("""
Cloudflare tunnel did not produce a URL.

Try:
1. Rerun only the tunnel cell.
2. Restart the Colab runtime and run all cells again.
3. Check that /usr/local/bin/cloudflared exists.
4. Check cloudflared --version.
5. Confirm local Flask health works:
   http://127.0.0.1:7860/health
""")
    raise RuntimeError("Cloudflare tunnel URL was not produced. Try rerunning this cell, or restart the Colab runtime.")
health_url = f"{public_url}/health"
print("Testing public health endpoint:", health_url)
public_health = None
for attempt in range(12):
    try:
        public_health = requests.get(health_url, timeout=20).json()
        break
    except Exception as exc:
        print(f"Public health check attempt {attempt + 1}/12 failed: {exc}")
        print("The trycloudflare DNS/route can take a short time to become reachable.")
        time.sleep(5)
if public_health is None:
    print("""
Public health check did not succeed yet, but the tunnel URL was created.
You can still try the COLAB_STT_URL below from your local machine.
If it fails locally, rerun only this tunnel cell to get a fresh URL.
""")
else:
    print(public_health)
print("\nColab Whisper worker is running.")
print(f"Set COLAB_STT_URL to: {public_url}/transcribe")
print("\nmacOS/Linux:")
print(f'export COLAB_STT_URL="{public_url}/transcribe"')
print("./run_web_app.sh")
print("\nWindows PowerShell:")
print(f'$env:COLAB_STT_URL="{public_url}/transcribe"')
print(".\\run_web_app.ps1")